In [1]:
import pandas as pd
from micom.workflows import build
from micom import load_pickle
from micom.qiime_formats import load_qiime_model_db, load_qiime_medium
from micom.workflows import grow
import pickle
from micom.viz import *
from micom.workflows import tradeoff

In [2]:
taxonomy_names = {'B_caccae': 'Bacteroides caccae',
                'B_cellulosilyticus_WH2': 'Bacteroides cellulosilyticus',
                'B_ovatus': 'Bacteroides ovatus',
                'B_thetaiotaomicron': 'Bacteroides thetaiotaomicron',
                'B_uniformis': 'Bacteroides uniformis',
                'B_vulgatus': 'Phocaeicola vulgatus',
                'C_aerofaciens': 'Collinsella aerofaciens',
                'C_scindens': 'Clostridium scindens',
                'C_spiroforme': 'Clostridium spiroforme',
                'D_longicatena': 'Dorea longicatena',
                'P_distasonis': 'Parabacteroides distasonis',
                'R_obeum': 'Blautia obeum',
                }

model_ids = {'B_caccae': 'Bacteroides_caccae_ATCC_43185.xml',
                'B_cellulosilyticus_WH2': 'Bacteroides_cellulosilyticus_WH2.xml',
                'B_ovatus': 'Bacteroides_ovatus_ATCC_8483.xml',
                'B_thetaiotaomicron': 'Bacteroides_thetaiotaomicron_VPI_5482.xml',
                'B_uniformis': 'Bacteroides_uniformis_ATCC_8492.xml',
                'B_vulgatus': 'Bacteroides_vulgatus_ATCC_8482.xml',
                'C_aerofaciens': 'Collinsella_aerofaciens_ATCC_25986.xml',
                'C_scindens': 'Clostridium_scindens_ATCC_35704.xml',
                'C_spiroforme': 'Clostridium_spiroforme_DSM_1552.xml',
                # 'D_longicatena': 'Dorea_longicatena',
                'P_distasonis': 'Parabacteroides_distasonis_ATCC_8503.xml',
                'R_obeum': 'Blautia_obeum_ATCC_29174.xml'}

In [3]:
hf0_reads = pd.read_table('../../ecological-modeling/mcnulty-datasets/HF0/reads.tsv', index_col=0, sep='\t')
hf0_reads = hf0_reads.loc[:,['-12' in col for col in hf0_reads.columns]]
hf0_reads.reset_index(inplace=True)
hf0_reads = hf0_reads.melt(id_vars=['name'], var_name='sample', value_name='reads')
hf0_reads['sample'] = hf0_reads['sample'].str.replace('-12', '')
hf0_reads['species'] = hf0_reads['name'].map(taxonomy_names)
hf0_reads.columns = ['id', 'sample_id', 'abundance', 'species']
hf0_reads['genus'] = hf0_reads['species'].str.split(' ').str[0]
hf0_reads = hf0_reads[hf0_reads['id'] != 'D_longicatena']
hf0_reads['file'] = 'models/Bacteria/' + hf0_reads['id'].map(model_ids)
hf0_reads['sample_id'] = 'mouse_' + hf0_reads['sample_id']
hf0_reads['diet'] = 'HF/HS'
hf0_reads.to_csv('hf0_data.tsv', sep='\t', index=False)
hf0_reads

,id,sample_id,abundance,species,genus,file,diet
0,B_caccae,mouse_08,37591.0,Bacteroides caccae,Bacteroides,models/Bacteria/Bacteroides_caccae_ATCC_43185.xml,HF/HS
1,B_cellulosilyticus_WH2,mouse_08,19336.2,Bacteroides cellulosilyticus,Bacteroides,models/Bacteria/Bacteroides_cellulosilyticus_W...,HF/HS
2,B_ovatus,mouse_08,2033.4,Bacteroides ovatus,Bacteroides,models/Bacteria/Bacteroides_ovatus_ATCC_8483.xml,HF/HS
3,B_thetaiotaomicron,mouse_08,15765.3,Bacteroides thetaiotaomicron,Bacteroides,models/Bacteria/Bacteroides_thetaiotaomicron_V...,HF/HS
4,B_uniformis,mouse_08,3533.1,Bacteroides uniformis,Bacteroides,models/Bacteria/Bacteroides_uniformis_ATCC_849...,HF/HS
...,...,...,...,...,...,...,...
90,C_aerofaciens,mouse_15,637.2,Collinsella aerofaciens,Collinsella,models/Bacteria/Collinsella_aerofaciens_ATCC_2...,HF/HS
91,C_scindens,mouse_15,2967.0,Clostridium scindens,Clostridium,models/Bacteria/Clostridium_scindens_ATCC_3570...,HF/HS
92,C_spiroforme,mouse_15,18.6,Clostridium spiroforme,Clostridium,models/Bacteria/Clostridium_spiroforme_DSM_155...,HF/HS
94,P_distasonis,mouse_15,5123.8,Parabacteroides distasonis,Parabacteroides,models/Bacteria/Parabacteroides_distasonis_ATC...,HF/HS


In [4]:
manifest_hf0 = build(
    hf0_reads, 
    model_db=None,
    out_folder='models/Community/hf0', 
    cutoff=0.000001, 
    threads=4
    )
manifest_hf0